<link rel="stylesheet" href="notebooks/styles.css">

<div class="title-wrap">
  <h1 class="title-main" style="font-weight: bold; font-size: 2.65rem; margin-bottom: 0.5rem;">
  Spatial Data Science Approaches to Wildfire Severity Modeling
</h1>
<h2 class="title-sub" style="font-style: italic; font-size: 1.8rem; margin-top: 0rem; margin-bottom: 0.2rem;">
  A GIS‑Driven, Tree‑Based Machine Learning Analysis of California Wildfires
</h2>
</div>

# Module 13: *Additional Models*

## Metadata

---
### Contents  
> 1. **
> 2. **
---
### Notes

A module to experiment with other models

---
### Inputs
- `X.csv` Model data
- `y.csv` Target data
- `details.csv` details reguarding data
- `pal_X`,`pal_details` data for 2025 predictions
- `best_strategy` best class balancing strategies calculated from module 06
- `model_parameters` optimum parameters as determined by module 07

---
### Outputs  
- `predictions.csv` Dataset containing prediction models composed of a categorical prediction 0,1,2 to be used for interpolation in ArcGIS.
---
### User Created Dependencies  

In [1]:
import os
import sys

# Allow import of custom modules from the parent directory
sys.path.append(os.path.abspath(os.path.join('..')))

from src.data_utils import *
from src.model_utils import *
from src.plot_utils import *

---
### Third Party Dependencies

In [2]:
# Core libraries
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

# Modeling libraries
from sklearn.ensemble import RandomForestClassifier
from sklearn.neighbors import KNeighborsClassifier
import xgboost as xgb
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import RidgeClassifier
from imblearn.over_sampling import SMOTE
from imblearn.under_sampling import RandomUnderSampler
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.metrics import mean_squared_error, r2_score
from sklearn.linear_model import LogisticRegression
from sklearn.preprocessing import MinMaxScaler

# Geospatial libraries
import geopandas as gpd
from shapely.geometry import Point

from datetime import timedelta
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, classification_report, confusion_matrix

In [3]:
from sklearn.tree import DecisionTreeClassifier, ExtraTreeClassifier
from sklearn.ensemble import (
    RandomForestClassifier,
    ExtraTreesClassifier,
    GradientBoostingClassifier,
    HistGradientBoostingClassifier
)

# Third-party sklearn-compatible models
from xgboost import XGBClassifier
from lightgbm import LGBMClassifier
from catboost import CatBoostClassifier

---
### Global Constants

In [4]:
# first day to analyze in weather dataset
FIRST_DATE = pd.to_datetime('2018-01-01').date()

# last day to analyze in weather dataset
LAST_DATE = pd.to_datetime('2024-12-31').date()

---

### Load Data

In [5]:
X = pd.read_csv('../data/processed/X_scaled.csv')
y = pd.read_csv('../data/processed/y_reduced.csv').squeeze()  # Load as Series
details = pd.read_csv('../data/processed/details.csv')

best_strategy = pd.read_csv('../data/processed/best_strategy.csv')
model_parameters = pd.read_csv('../data/processed/model_parameters.csv', index_col=0)

pal_details = pd.read_csv('../data/processed/pal_details.csv')
pal_X = pd.read_csv('../data/processed/pal_X.csv')
details['Date'] = pd.to_datetime(details['Date']).dt.date

In [6]:
final = pd.concat([details, X, y], axis=1)

In [7]:
final['Date'] = pd.to_datetime(final['Date'])

# Number of rows to keep from class 0
n_keep = 5000

# Split by class
class0 = final[final['Target'] == 0]
class1 = final[final['Target'] == 1]
class2 = final[final['Target'] == 2]

# Sample class 0 down to 50,000 rows
class0_sampled = class0.sample(n=n_keep, random_state=14)
class1_sampled = class1.sample(n=n_keep, random_state=14)
class2_sampled = class2.sample(n=n_keep, random_state=14)
# Combine back together
reduced = pd.concat([class0_sampled, class1_sampled , class2_sampled], ignore_index=True)

In [8]:
reduced = reduced.drop(columns='Year')

In [9]:
# Columns to drop for feature interaction analysis
text_columns = ['Sample_ID', 'Date','Target', 'grid_id', 'centroid_easting', 'centroid_northing',
       'geometry', 'fire_count', 'total_fire_damage','acres',  'dominant_section_description',
                'dominant_province_description','maximum_x', 'minimum_y',
       'maximum_y', 'minimum_x','area_in_cali']

y = reduced['Target']
X = reduced.drop(columns=text_columns)
details = reduced[text_columns]

In [10]:
y.value_counts()

Target
0.0    5000
1.0    5000
2.0    5000
Name: count, dtype: int64

In [11]:
model_parameters = pd.read_csv('../data/processed/model_parameters.csv', index_col=0)

In [12]:
RF_parameters = model_parameters.loc['RandomForest'].dropna().to_dict()
XGB_parameters = model_parameters.loc['XGBoost'].dropna().to_dict()
LGBM_parameters = model_parameters.loc['LGBM'].dropna().to_dict()

optimal_learning_rate = XGB_parameters['learning_rate']
optimal_learning_rate_lgbm = LGBM_parameters['learning_rate']

In [13]:
# Helper function to convert to int if possible
def convert_to_int(d):
    return {k: int(float(v)) if str(v).replace('.', '', 1).isdigit() else v for k, v in d.items()}

RF_parameters = convert_to_int(RF_parameters)
XGB_parameters = convert_to_int(XGB_parameters)
LGBM_parameters = convert_to_int(LGBM_parameters)

XGB_parameters['learning_rate'] = optimal_learning_rate
LGBM_parameters['learning_rate'] = optimal_learning_rate_lgbm
LGBM_parameters['verbose'] = -1

In [14]:
# Build Final tuned models
optimum_xgb_model = xgb.XGBClassifier(**XGB_parameters)
optimum_rf = RandomForestClassifier(**RF_parameters)
optimum_lgbm_model = LGBMClassifier(**LGBM_parameters)

## 1. Build Models

## 2. Train Models

In [15]:
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=14)

# Extra Models

In [17]:
optimum_lgbm_model.fit(X_train, y_train)
pred = optimum_lgbm_model.predict(X_test)
print("LGBM:\n", classification_report(y_test, pred))

LGBM:
               precision    recall  f1-score   support

         0.0       0.76      0.72      0.74      1006
         1.0       0.76      0.79      0.77      1016
         2.0       0.70      0.71      0.71       978

    accuracy                           0.74      3000
   macro avg       0.74      0.74      0.74      3000
weighted avg       0.74      0.74      0.74      3000



In [18]:
optimum_xgb_model.fit(X_train, y_train)
pred = optimum_xgb_model.predict(X_test)
print("XGBoost:\n", classification_report(y_test, pred))

XGBoost:
               precision    recall  f1-score   support

         0.0       0.76      0.72      0.74      1006
         1.0       0.77      0.81      0.79      1016
         2.0       0.71      0.71      0.71       978

    accuracy                           0.75      3000
   macro avg       0.75      0.75      0.75      3000
weighted avg       0.75      0.75      0.75      3000



In [19]:
# Create a Series for easy sorting
importances = pd.Series(optimum_xgb_model.feature_importances_, index=X.columns)

# Get top 10
top10 = importances.sort_values(ascending=False).head(10)

top10.to_frame(name="XHBoost Top 10 Importance")

,XHBoost Top 10 Importance
total_population,0.159039
total_housing,0.111352
population_density,0.036230
section_code,0.031173
Season,0.030542
road_length_meters,0.029863
Month,0.028824
2-Year Avg Fires,0.025931
wetlands_percent,0.024084
intermix_zone,0.023907


In [20]:
optimum_rf.fit(X_train, y_train)
pred = optimum_rf.predict(X_test)
print("Random Forest:\n", classification_report(y_test, pred))

Random Forest:
               precision    recall  f1-score   support

         0.0       0.74      0.73      0.74      1006
         1.0       0.75      0.75      0.75      1016
         2.0       0.66      0.67      0.66       978

    accuracy                           0.72      3000
   macro avg       0.72      0.72      0.72      3000
weighted avg       0.72      0.72      0.72      3000



In [21]:
# Create a Series for easy sorting
importances = pd.Series(optimum_rf.feature_importances_, index=X.columns)

# Get top 10
top10 = importances.sort_values(ascending=False).head(10)

top10.to_frame(name="Random Forest Top 10 Importance")

,Random Forest Top 10 Importance
2-Year Avg Fires,0.064231
Palmer Drought Severity Index,0.034120
road_density,0.032656
population_density,0.031834
total_housing,0.028712
Solar Radiation,0.027827
median_income,0.027759
total_population,0.027585
road_length_meters,0.026844
1000-hour Dead Fuel Moisture,0.025377


In [22]:
extra_trees = ExtraTreesClassifier(
    n_estimators=300,
    max_depth=None,
    min_samples_split=2,
    class_weight="balanced_subsample",
    random_state=42
)

extra_trees.fit(X_train, y_train)
pred = extra_trees.predict(X_test)
print("Extra Trees:\n", classification_report(y_test, pred))

Extra Trees:
               precision    recall  f1-score   support

         0.0       0.74      0.73      0.74      1006
         1.0       0.74      0.73      0.73      1016
         2.0       0.64      0.66      0.65       978

    accuracy                           0.71      3000
   macro avg       0.71      0.71      0.71      3000
weighted avg       0.71      0.71      0.71      3000



In [23]:
# Create a Series for easy sorting
importances = pd.Series(extra_trees.feature_importances_, index=X.columns)

# Get top 10
top10 = importances.sort_values(ascending=False).head(10)

top10.to_frame(name="Extra Trees Top 10 Importance")

,Extra Trees Top 10 Importance
2-Year Avg Fires,0.043720
Solar Radiation 7 Day Avg,0.026915
Palmer Drought Severity Index,0.026738
Solar Radiation,0.022971
1000-hour Dead Fuel Moisture,0.022879
Energy Release Component,0.022383
Standardized Precipitation Evapotranspiration Index 180-Day,0.022129
Standardized Precipitation Index 180-Day,0.021560
Daily Maximum Air Temperature 7 Day Avg,0.021097
Month,0.021080


In [24]:
gboost = GradientBoostingClassifier(
    n_estimators=200,
    learning_rate=0.1,
    max_depth=3
)

gboost.fit(X_train, y_train)
pred = gboost.predict(X_test)
print("GradientBoosting:\n", classification_report(y_test, pred))

GradientBoosting:
               precision    recall  f1-score   support

         0.0       0.76      0.72      0.74      1006
         1.0       0.76      0.80      0.78      1016
         2.0       0.71      0.71      0.71       978

    accuracy                           0.74      3000
   macro avg       0.74      0.74      0.74      3000
weighted avg       0.74      0.74      0.74      3000



In [25]:
# Create a Series for easy sorting
importances = pd.Series(gboost.feature_importances_, index=X.columns)

# Get top 10
top10 = importances.sort_values(ascending=False).head(10)

top10.to_frame(name="Gradient Boost Top 10 Importance")

,Gradient Boost Top 10 Importance
2-Year Avg Fires,0.191717
road_length_meters,0.076407
total_population,0.069347
Solar Radiation 7 Day Avg,0.049573
section_code,0.048147
Palmer Drought Severity Index,0.047713
median_income,0.040912
intermix_zone,0.040675
total_housing,0.039534
1000-hour Dead Fuel Moisture,0.031165


In [26]:
hgb = HistGradientBoostingClassifier(
    learning_rate=0.05,
    max_depth=10,
    max_iter=500
)

hgb.fit(X_train, y_train)
pred = hgb.predict(X_test)
print("HistGradientBoosting:\n", classification_report(y_test, pred))

HistGradientBoosting:
               precision    recall  f1-score   support

         0.0       0.76      0.72      0.74      1006
         1.0       0.77      0.81      0.79      1016
         2.0       0.72      0.71      0.71       978

    accuracy                           0.75      3000
   macro avg       0.75      0.75      0.75      3000
weighted avg       0.75      0.75      0.75      3000



from sklearn.inspection import permutation_importance

r = permutation_importance(hgb, X, y, n_repeats=10, random_state=42)

importances = pd.Series(r.importances_mean, index=X.columns)
top10 = importances.sort_values(ascending=False).head(10)


top10.to_frame(name="Histogram Gradient Boost Top 10 Importance")

In [27]:
lgbm = LGBMClassifier(
    n_estimators=500,
    num_leaves=31,
    max_depth=-1,
    learning_rate=0.05,
    class_weight='balanced'
)

lgbm.fit(X_train, y_train)
pred = lgbm.predict(X_test)
print("LightGBM:\n", classification_report(y_test, pred))

LightGBM:
               precision    recall  f1-score   support

         0.0       0.76      0.72      0.74      1006
         1.0       0.76      0.80      0.78      1016
         2.0       0.71      0.70      0.71       978

    accuracy                           0.74      3000
   macro avg       0.74      0.74      0.74      3000
weighted avg       0.74      0.74      0.74      3000



In [28]:
# Create a Series for easy sorting
importances = pd.Series(lgbm.feature_importances_, index=X.columns)

# Get top 10
top10 = importances.sort_values(ascending=False).head(10)

top10.to_frame(name="LGBM Top 10 Importance")

,LGBM Top 10 Importance
2-Year Avg Fires,2671
Palmer Drought Severity Index,1981
Wind Speed 7 Day Avg,1465
Solar Radiation 7 Day Avg,1417
Solar Radiation,1274
Wind Speed,1240
Daily Minimum Air Temperature,1179
Minimum Relative Humidity,1176
1000-hour Dead Fuel Moisture,1171
Specific Humidity,1150


In [29]:
cat = CatBoostClassifier(
    iterations=500,
    depth=8,
    learning_rate=0.05,
    loss_function='MultiClass',
    verbose=False
)

cat.fit(X_train, y_train)
pred = cat.predict(X_test)
print("CatBoost:\n", classification_report(y_test, pred))

CatBoost:
               precision    recall  f1-score   support

         0.0       0.76      0.73      0.74      1006
         1.0       0.77      0.79      0.78      1016
         2.0       0.70      0.70      0.70       978

    accuracy                           0.74      3000
   macro avg       0.74      0.74      0.74      3000
weighted avg       0.74      0.74      0.74      3000



In [30]:
# Create a Series for easy sorting
importances = pd.Series(cat.get_feature_importance(), index=X.columns)

# Get top 10
top10 = importances.sort_values(ascending=False).head(10)

top10.to_frame(name="Cat Top 10 Importance")


,Cat Top 10 Importance
2-Year Avg Fires,14.420078
Palmer Drought Severity Index,4.772207
section_code,4.658575
Month,3.654632
Solar Radiation 7 Day Avg,3.323769
elevation_mean,2.428212
Standardized Precipitation Index 180-Day,2.242219
Solar Radiation,2.118354
median_income,2.082309
road_length_meters,2.076979


In [31]:
dt = DecisionTreeClassifier(
    max_depth=None,
    class_weight="balanced"
)

dt.fit(X_train, y_train)
pred = dt.predict(X_test)
print("DecisionTree:\n", classification_report(y_test, pred))

DecisionTree:
               precision    recall  f1-score   support

         0.0       0.65      0.62      0.64      1006
         1.0       0.66      0.67      0.67      1016
         2.0       0.58      0.61      0.59       978

    accuracy                           0.63      3000
   macro avg       0.63      0.63      0.63      3000
weighted avg       0.63      0.63      0.63      3000



In [32]:
importances = pd.Series(dt.feature_importances_, index=X.columns)

# Get top 10
top10 = importances.sort_values(ascending=False).head(10)

top10.to_frame(name="Decision Tree Top 10 Importance")


,Decision Tree Top 10 Importance
2-Year Avg Fires,0.094102
total_housing,0.089428
Palmer Drought Severity Index,0.038240
Solar Radiation 7 Day Avg,0.036036
section_code,0.034671
Month,0.033809
Wind Speed 7 Day Avg,0.028556
median_income,0.027268
Solar Radiation,0.025658
Actual Evapotranspiration,0.025646


In [33]:
et = ExtraTreeClassifier(
    max_depth=None,
    class_weight="balanced"
)

et.fit(X_train, y_train)
pred = et.predict(X_test)
print("ExtraTree:\n", classification_report(y_test, pred))

ExtraTree:
               precision    recall  f1-score   support

         0.0       0.64      0.65      0.64      1006
         1.0       0.60      0.59      0.60      1016
         2.0       0.52      0.53      0.53       978

    accuracy                           0.59      3000
   macro avg       0.59      0.59      0.59      3000
weighted avg       0.59      0.59      0.59      3000



In [34]:
# Create a Series for easy sorting
importances = pd.Series(et.feature_importances_, index=X.columns)

# Get top 10
top10 = importances.sort_values(ascending=False).head(10)

top10.to_frame(name="Extra Tree Top 10 Importance")

,Extra Tree Top 10 Importance
power_line_density,0.048736
2-Year Avg Fires,0.042732
Energy Release Component,0.039675
median_income,0.035619
Vapor Pressure Deficit 7 Day Avg,0.028905
Solar Radiation 7 Day Avg,0.028649
Standardized Precipitation Evapotranspiration Index 180-Day,0.024119
Standardized Precipitation Evapotranspiration Index 90-Day,0.023546
interface_zone,0.022365
100-hour Dead Fuel Moisture,0.022242


### Interpolation

<img src="../data/maps/IDW_RF.jpg" width="600">

## 3. Export File